# Data Science & AI Bootcamp
**USF INFORMS Student Chapter** · Sponsored by the **INFORMS Quality, Statistics, and Reliability (QSR) Section**
Friday, October 9, 2026 · ENG 3

---

## Part 1: Foundations with Python, using Fitbit data
*Part 2 (AI & data science in QSR) is in the slides, and Part 3 (PyTorch demos) has its own notebook.*

### Our question: *Do more active days come with better sleep?*

Fitness apps tell us to hit **10,000 steps** and to get **7+ hours of sleep**. Are those two goals connected?
We'll answer this with real wearable data, following the same workflow data scientists use every day:

| Step | What we do | ~Time |
|---|---|---|
| **1. Load** | Read the data, understand what one row means, combine tables | 5 min |
| **2. Explore** | Check distributions, visualize relationships, deal with outliers | 10 min |
| **3. Test** | State a hypothesis, run several statistical tests, interpret them carefully | 10 min |

> **How to use this notebook:** run cells top to bottom with **Shift + Enter**.
> 🔍 boxes are questions to discuss; click **Pointers** to check your thinking.

## 0. Setup
Four standard libraries, all preinstalled in Colab:
**pandas** (tables) · **numpy** (math) · **matplotlib** (plots) · **scipy.stats** (statistical tests)

Running this notebook **outside Colab** (e.g., Jupyter on your laptop)? Uncomment the install line below and run it once.

In [ ]:
# Only needed outside Colab: remove the "#" and run once
# !pip install pandas numpy matplotlib scipy

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

plt.rcParams["figure.figsize"] = (8, 4)
GREEN, GOLD = "#006747", "#CFC493"

BASE_URL = "https://raw.githubusercontent.com/informs-sc-usf/bootcamp-2026/main/data/"

---
## 1. Loading the data

About 30 people shared one month of Fitbit data (April 12 – May 12, 2016) through an online survey.
*Source: Furberg, Brinton, Keating & Ortiz (2016), Zenodo, doi:10.5281/zenodo.53894.*

We'll use two tables:
- **Daily activity:** one row = one person on one day (steps, active minutes, calories)
- **Sleep:** one row = one person's sleep on one date (minutes asleep, minutes in bed)

In [ ]:
daily = pd.read_csv(BASE_URL + "dailyActivity_merged.csv")
sleep = pd.read_csv(BASE_URL + "sleepDay_merged.csv")

print("daily:", daily.shape, "  sleep:", sleep.shape)   # (rows, columns)
daily.head()

### First checks: data types and duplicates
`.info()` lists each column's type. Dates often load as plain **text**, which Python can't sort or compare correctly, so we convert them.

In [ ]:
daily.info()

In [ ]:
daily["Date"] = pd.to_datetime(daily["ActivityDate"], format="%m/%d/%Y")
sleep["Date"] = pd.to_datetime(sleep["SleepDay"], format="%m/%d/%Y %I:%M:%S %p")

print("People:", daily["Id"].nunique(), "| Dates:", daily["Date"].min().date(), "to", daily["Date"].max().date())
print("Duplicate sleep rows:", sleep.duplicated().sum())
sleep = sleep.drop_duplicates()

### Combining the tables
To compare activity and sleep, we need them in **one table**. We **merge** on the two columns that identify a row: person (`Id`) and `Date`.
We also convert sleep to hours, which is easier to read than minutes.

In [ ]:
df = daily.merge(sleep, on=["Id", "Date"], how="inner")
df["SleepHours"] = df["TotalMinutesAsleep"] / 60

print(f"{len(df)} days with both activity and sleep, from {df['Id'].nunique()} people")
df[["Id", "Date", "TotalSteps", "SleepHours", "TotalTimeInBed"]].head()

> 🔍 **Think about it**

1. We started with 940 activity days but only have ~410 rows now. Where did the rest go?
2. `how="inner"` keeps only matching rows. Could the people who log sleep be **different** from those who don't?


<details><summary><b>Pointers (click to expand)</b></summary>


1. An inner merge keeps only days that appear in **both** tables. Many people never logged sleep (only 24 of 33 did), so their days drop out.
2. Yes. People who wear their tracker to bed may be more health-conscious, for example. Our conclusions apply to **sleep-trackers**, not everyone. This is called **selection bias**, and it's worth stating whenever you merge or filter.

</details>

#### Bonus: what "big data" looks like
The same study also recorded heart rate every 5–15 seconds. That file has **~2.5 million rows**. We won't use it for our question, but run this to see how pandas handles it.

In [ ]:
hr = pd.read_csv(BASE_URL + "heartrate_seconds_merged.csv.gz")   # ~10 seconds
print(f"{len(hr):,} heart-rate readings from {hr['Id'].nunique()} people, "
      f"{hr.memory_usage(deep=True).sum()/1e6:.0f} MB in memory")

---
## 2. Exploring the data
Before testing anything, **look at the data**. Three questions:
1. What do the values look like? (**distributions**)
2. How do the variables relate? (**visualization**)
3. Is anything strange or suspicious? (**outliers**)

### 2.1 Checking distributions
`.describe()` gives the classic summary: count, mean, standard deviation, min, quartiles, and max.

In [ ]:
df[["TotalSteps", "SleepHours", "TotalTimeInBed"]].describe().round(1)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

axes[0].hist(df["TotalSteps"], bins=30, color=GREEN, edgecolor="white")
axes[0].axvline(10000, color="black", ls="--", label="10,000 steps")
axes[0].set(title="Daily steps", xlabel="Steps", ylabel="Days"); axes[0].legend()

axes[1].hist(df["SleepHours"], bins=30, color=GOLD, edgecolor="white")
axes[1].axvline(7, color="black", ls="--", label="7 hours")
axes[1].set(title="Sleep per night", xlabel="Hours asleep"); axes[1].legend()
plt.tight_layout(); plt.show()

Many statistical tests assume the data are **normally distributed** (bell-shaped). Two ways to check:
- **Q-Q plot:** if the data are normal, the points fall on the straight line.
- **Shapiro–Wilk test:** a small p-value (< 0.05) is evidence the data are **not** normal.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, col in zip(axes, ["TotalSteps", "SleepHours"]):
    stats.probplot(df[col], dist="norm", plot=ax)
    ax.set_title(f"Q-Q plot: {col}")
    ax.get_lines()[0].set(color=GREEN, markersize=3); ax.get_lines()[1].set(color="black")
plt.tight_layout(); plt.show()

for col in ["TotalSteps", "SleepHours"]:
    print(f"Shapiro-Wilk {col}: p = {stats.shapiro(df[col]).pvalue:.1e}")

> 🔍 **Think about it**

1. Which variable departs more from a bell shape? Where do the Q-Q points bend away from the line?
2. Both Shapiro–Wilk p-values are tiny. Does that mean we can't use any test that assumes normality?


<details><summary><b>Pointers (click to expand)</b></summary>


1. Steps are fairly symmetric here, with only mild bending at the ends of the Q-Q plot. Sleep is bell-shaped in the middle but has a **long left tail**: a cluster of very short "nights" (1–3 hours) that pulls the Q-Q points far below the line on the left.
2. Not necessarily. With hundreds of data points, Shapiro–Wilk flags even small departures from normality. Also, the t-test compares **means**, and means of large samples are approximately normal anyway (the **Central Limit Theorem**). The safe habit: run a test that **doesn't** assume normality too, and see if the conclusion changes. We'll do exactly that in Section 3.

</details>

### 2.2 Visualizing the relationship
Now the question itself: **do steps and sleep move together?** A scatter plot shows every day as one dot.

In [ ]:
plt.scatter(df["TotalSteps"], df["SleepHours"], color=GREEN, alpha=0.5)
plt.axvline(10000, color="gray", ls="--"); plt.axhline(7, color="gray", ls="--")
plt.xlabel("Daily steps"); plt.ylabel("Hours asleep"); plt.title("Each dot is one person-day")
plt.show()

Let's split days into two groups, **active** (10,000+ steps) and **less active**, and compare their sleep with a **boxplot**.
The box holds the middle 50% of days, the line inside is the **median**, and dots beyond the whiskers are potential **outliers**.

In [ ]:
df["Active"] = df["TotalSteps"] >= 10000

plt.boxplot([df.loc[~df["Active"], "SleepHours"], df.loc[df["Active"], "SleepHours"]],
            patch_artist=True, boxprops=dict(facecolor=GOLD), medianprops=dict(color=GREEN, lw=2))
plt.xticks([1, 2], ["< 10,000 steps", "10,000+ steps"])
plt.ylabel("Hours asleep"); plt.title("Sleep on less active vs. active days"); plt.show()

print(df.groupby("Active")["SleepHours"].agg(["count", "mean", "median"]).round(2))

> 🔍 **Think about it**

1. Based on the plots, do active days seem to come with **more** or **less** sleep?
2. What do you notice at the **bottom** of both boxes?


<details><summary><b>Pointers (click to expand)</b></summary>


1. Surprisingly, **less**: active days have a lower median sleep. This is the opposite of what many people expect, so we'll need to test it carefully.
2. Both groups have a cluster of dots far below the box, at around 1–3 hours. That's our next topic.

</details>

### 2.3 Outliers: error, or real?
A common rule flags values beyond **1.5 × IQR** (interquartile range) from the box edges.
But an outlier rule only tells you **where to look**. You decide what to do by understanding **why** the value is unusual.

In [ ]:
q1, q3 = df["SleepHours"].quantile([0.25, 0.75])
iqr = q3 - q1
low, high = q1 - 1.5 * iqr, q3 + 1.5 * iqr
outliers = df[(df["SleepHours"] < low) | (df["SleepHours"] > high)]

print(f"IQR rule: flag sleep below {low:.1f} h or above {high:.1f} h  ->  {len(outliers)} flagged nights")
outliers[["Id", "Date", "SleepHours", "TotalTimeInBed", "TotalSleepRecords"]].sort_values("SleepHours").head(8)

> 🔍 **Think about it**

1. Look at `TotalTimeInBed` for the shortest nights. What do you think happened on those nights?
2. Should we **delete** these rows, **keep** them, or **fix** them? What would you write in a report to justify the choice?


<details><summary><b>Pointers (click to expand)</b></summary>


1. Time in bed is only about 1 hour, so these are most likely **naps**, a tracker that ran out of battery, or a band taken off during the night. They are not real full nights of sleep.
2. Since our question is about **nightly** sleep, it is reasonable to exclude records shorter than 3 hours, **and say so**. The very long nights (11+ hours) are plausible (weekends, illness), so we keep them. Rule of thumb: remove values that are **measurement problems**, keep values that are **real but unusual**, and always document both.

</details>

In [ ]:
clean = df[df["SleepHours"] >= 3].copy()
print(f"Kept {len(clean)} of {len(df)} nights ({len(df) - len(clean)} likely naps or partial records removed)")

---
## 3. Hypothesis testing
### State the hypothesis *before* running tests
- **H₀ (null):** sleep is the **same** on active (10,000+ steps) and less active days.
- **H₁ (alternative):** sleep **differs** between them.

We'll use a significance level of **α = 0.05**. A **p-value** is the probability of seeing a difference at least this large **if H₀ were true**. A small p-value means the data would be surprising under H₀.

Different tests answer slightly different versions of the question, so we'll run several and compare.

### 3.1 Welch's t-test: do the **means** differ?
Compares two group averages without assuming equal spread.

In [ ]:
active = clean.loc[clean["Active"], "SleepHours"]
less   = clean.loc[~clean["Active"], "SleepHours"]

t = stats.ttest_ind(active, less, equal_var=False)
print(f"Mean sleep: active days {active.mean():.2f} h vs. less active days {less.mean():.2f} h")
print(f"Difference: {active.mean() - less.mean():+.2f} h  |  t = {t.statistic:.2f}, p = {t.pvalue:.1e}")

### 3.2 Mann–Whitney U test: the no-normality-needed version
Works on **ranks** instead of raw values, so it doesn't assume a bell curve and is less sensitive to outliers.

In [ ]:
u = stats.mannwhitneyu(active, less)
print(f"Median sleep: active {active.median():.2f} h vs. less active {less.median():.2f} h  |  p = {u.pvalue:.1e}")

### 3.3 Correlation: is it a trend, not just two groups?
Splitting at 10,000 steps is arbitrary. **Spearman's ρ** measures whether more steps go with more (or less) sleep across the **whole range**, using ranks.

In [ ]:
rho = stats.spearmanr(clean["TotalSteps"], clean["SleepHours"])
print(f"Spearman rho = {rho.statistic:.2f}, p = {rho.pvalue:.1e}")

### 3.4 Chi-square test: do the **goals** go together?
Turn both variables into yes/no: *hit 10,000 steps?* and *slept 7+ hours?* Then check whether the two are independent.

In [ ]:
clean["Slept7"] = clean["SleepHours"] >= 7
table = pd.crosstab(clean["Active"], clean["Slept7"])
table.index, table.columns = ["< 10k steps", "10k+ steps"], ["< 7 h sleep", "7+ h sleep"]
print(table, "\n")
chi2, p, dof, expected = stats.chi2_contingency(table)
print(f"Chi-square = {chi2:.1f}, p = {p:.3f}")
print("Share of days with 7+ h sleep:", (table["7+ h sleep"] / table.sum(axis=1)).round(2).to_dict())

> 🔍 **Think about it**

Four tests, all with small p-values. Does that mean we've **proven** that walking more makes you sleep less?


<details><summary><b>Pointers (click to expand)</b></summary>


No, for two reasons:
- **Correlation is not causation.** A busy day (travel, deadlines, events) could cause *both* more walking *and* less sleep.
- **Our tests assumed every day is independent**, but each person contributes many days. If some people simply walk a lot *and* sleep little in general, the pooled tests mix up differences **between people** with changes **within a person**. Let's fix that next.

</details>

### 3.5 The twist: compare each person with **themselves**
For each person, compute their average sleep on active days and on less active days. Then use a **paired test**, which looks at each person's own difference.

In [ ]:
per_person = clean.groupby(["Id", "Active"])["SleepHours"].mean().unstack().dropna()
per_person.columns = ["Less active days", "Active days"]
diff = per_person["Active days"] - per_person["Less active days"]

print(f"{len(per_person)} people had both kinds of days")
print(f"Average within-person difference: {diff.mean():+.2f} h  ({(diff < 0).sum()} of {len(diff)} slept less on active days)")
print(f"Paired t-test:        p = {stats.ttest_rel(per_person['Active days'], per_person['Less active days']).pvalue:.3f}")
print(f"Wilcoxon signed-rank: p = {stats.wilcoxon(per_person['Active days'], per_person['Less active days']).pvalue:.3f}")

for _, row in per_person.iterrows():
    plt.plot([0, 1], row.values, color=GREEN, alpha=0.6, marker="o")
plt.xticks([0, 1], per_person.columns); plt.xlim(-0.3, 1.3)
plt.ylabel("Average hours asleep"); plt.title("Each line is one person"); plt.show()

### Putting it all together

In [ ]:
#@title Summary table of all tests (run me)
summary = pd.DataFrame([
    ["Welch t-test",       "Do mean sleep hours differ?",                 "All days pooled",  t.pvalue],
    ["Mann-Whitney U",     "Do distributions differ (rank-based)?",       "All days pooled",  u.pvalue],
    ["Spearman rho",       "Do more steps go with less/more sleep?",      "All days pooled",  rho.pvalue],
    ["Chi-square",         "Are the 10k-step and 7-hour goals linked?",   "All days pooled",  p],
    ["Paired t-test",      "Does the same person sleep differently?",     f"{len(per_person)} people", stats.ttest_rel(per_person['Active days'], per_person['Less active days']).pvalue],
    ["Wilcoxon signed-rank","Same, rank-based",                           f"{len(per_person)} people", stats.wilcoxon(per_person['Active days'], per_person['Less active days']).pvalue],
], columns=["Test", "Question it answers", "Unit of analysis", "p-value"])
summary["Significant at 0.05?"] = np.where(summary["p-value"] < 0.05, "Yes", "No")
summary["p-value"] = summary["p-value"].map(lambda v: f"{v:.1e}" if v < 0.001 else f"{v:.3f}")
summary

> 🔍 **Think about it**

1. Why is the evidence much **weaker** in the paired tests than in the pooled ones?
2. In one or two sentences, how would you answer our original question: *do more active days come with better sleep?*
3. Look back at the merge: does a sleep record dated April 20 describe the night **before** or **after** that day's steps? Why does it matter?


<details><summary><b>Pointers (click to expand)</b></summary>


1. The paired tests use **one difference per person** (about 14 numbers instead of ~390 days), so they have less **statistical power**. They also remove differences **between** people, which were driving part of the pooled result. The direction is the same (most people sleep a bit less on active days), but the evidence is borderline: the two paired tests even land on opposite sides of 0.05.
2. Something like: *"In this sample, days with 10,000+ steps came with roughly 45–50 minutes **less** sleep overall, and about half that within the same person. This is an association, not proof of cause, and it comes from a small, self-selected group."*
3. The data don't tell us for sure. Depending on how Fitbit dates a sleep record, the "same date" may pair a day's steps with the **previous** night's sleep. If so, the question we actually tested is "do people walk more after a short night?", which is a different story. **Knowing how the data were recorded is part of the analysis.**

</details>

### Key takeaways
- **Load and check:** know what one row means; convert types; watch what a merge drops.
- **Explore before testing:** distributions, plots, and outliers shape which tests are appropriate.
- **Outliers need a reason:** remove measurement problems, keep real extremes, document both.
- **Match the test to the question:** means vs. ranks vs. trends vs. categories vs. paired designs.
- **A small p-value is not the end:** ask about independence, confounding, and how the data were collected.

> These are the building blocks of **Quality, Statistics, and Reliability**: trusting measurements, modeling variation, and drawing careful conclusions from data.

---
## Keep going with QSR
- **Join INFORMS QSR:** https://connect.informs.org/qsr/join-qsr3
- **QSR home** (webinars, tutorials, student competitions): https://connect.informs.org/qsr/home
- **USF INFORMS Student Chapter:** Instagram @informs_sc_usf · LinkedIn: informs-sc-usf

*Data: Furberg, R., Brinton, J., Keating, M., & Ortiz, A. (2016). Crowd-sourced Fitbit datasets 03.12.2016–05.12.2016. Zenodo. https://doi.org/10.5281/zenodo.53894*